# Music Store Customer Support

This notebook demonstrates the PDF workflow: customer verification, Chinook database access, mixed music and invoice routing, human-in-the-loop resume, and preference memory. Business logic remains in `src/music_support`.

## Graph sequence

`verify_info -> load_memory -> supervisor -> music_catalog/invoice_information -> create_memory`

Missing or invalid credentials route through `human_input` and resume on the same checkpointed thread.

In [1]:
from langgraph.types import Command
from music_support.app import create_support_app
from music_support.database import get_engine_for_chinook_db, required_tables

engine = get_engine_for_chinook_db()
required_tables(engine)

/home/mohamed_hisham/AI/Music-store-customer-support/.venv/lib/python3.14/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


{'Album',
 'Artist',
 'Customer',
 'Employee',
 'Genre',
 'Invoice',
 'InvoiceLine',
 'MediaType',
 'Playlist',
 'PlaylistTrack',
 'Track'}

In [2]:
app = create_support_app()
config = {'configurable': {'thread_id': 'notebook-demo'}}
result = app.invoke(
    {'messages': [('user', 'My phone number is +55 (12) 3923-5555. How much was my most recent purchase? What albums do you have by The Rolling Stones?')]},
    config,
)
print(result['final_response'])

I've found the information you requested:

Music catalog results for The Rolling Stones:
- artist: The Rolling Stones, album: Hot Rocks, 1964-1971 (Disc 1)
- artist: The Rolling Stones, album: No Security
- artist: The Rolling Stones, album: Voodoo Lounge

Your most recent purchase was on 2025-08-07 00:00:00, for a total amount of $8.91.

If you need any more details, feel free to ask!


In [3]:
new_app = create_support_app()
missing_config = {'configurable': {'thread_id': 'notebook-missing-credentials'}}
paused = new_app.invoke(
    {'messages': [('user', 'How much was my most recent purchase?')]},
    missing_config,
)
print(paused['__interrupt__'][0].value)
resumed = new_app.invoke(Command(resume='Customer ID: 1'), missing_config)
print(resumed['final_response'])

Please provide your customer ID, email, or phone number.
I've found the information you requested:

Your most recent purchase was on 2025-08-07 00:00:00, for a total amount of $8.91.

If you need any more details, feel free to ask!


In [4]:
follow_up = new_app.invoke(
    {'messages': [('user', 'List some songs that match my preferences?')]},
    missing_config,
)
print(follow_up['final_response'])

I've found the information you requested:

Songs matching your preferences:
- artist: The Rolling Stones, album: Hot Rocks, 1964-1971 (Disc 1), track: Time Is On My Side
- artist: The Rolling Stones, album: Hot Rocks, 1964-1971 (Disc 1), track: Heart Of Stone
- artist: The Rolling Stones, album: Hot Rocks, 1964-1971 (Disc 1), track: Play With Fire
- artist: The Rolling Stones, album: Hot Rocks, 1964-1971 (Disc 1), track: Satisfaction
- artist: The Rolling Stones, album: Hot Rocks, 1964-1971 (Disc 1), track: As Tears Go By
- artist: The Rolling Stones, album: Hot Rocks, 1964-1971 (Disc 1), track: Get Off Of My Cloud
- artist: The Rolling Stones, album: Hot Rocks, 1964-1971 (Disc 1), track: Mother's Little Helper
- artist: The Rolling Stones, album: Hot Rocks, 1964-1971 (Disc 1), track: 19th Nervous Breakdown
- artist: The Rolling Stones, album: Hot Rocks, 1964-1971 (Disc 1), track: Paint It Black
- artist: The Rolling Stones, album: Hot Rocks, 1964-1971 (Disc 1), track: Under My Thumb
-